<a href="https://colab.research.google.com/github/microsoft/qlib/blob/main/examples/workflow_by_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## How this notebook works (big picture)

Canonical qlib workflow in code: **train a LightGBM model on CSI300 stocks -> predict daily scores -> backtest a top-k portfolio -> visualize results**.

```
Alpha158 features --> LGBModel --> pred scores --> TopkDropoutStrategy --> backtest
(158 tech factors)   (2008-2014)  (2017-2020)     (top50, swap 5/day)    (costs+limits)
                                                                  |
                                   report / risk / IC / performance charts
```

Local run notes: use a kernel that has **pyqlib with compiled extensions** installed (e.g. `D:/ProgramData/miniconda3/envs/qlib`), and run from a directory that does NOT shadow the installed package. First run computes Alpha158 features for ~300 stocks x 12 years (slow); results are cached on disk so later runs are fast.

In [ ]:
#  Copyright (c) Microsoft Corporation.
#  Licensed under the MIT License.

In [ ]:
import sys, site
from pathlib import Path

################################# NOTE #################################
#  Please be aware that if colab installs the latest numpy and pyqlib  #
#  in this cell, users should RESTART the runtime in order to run the  #
#  following cells successfully.                                       #
########################################################################

try:
    import qlib
except ImportError:
    # install qlib
    ! pip install --upgrade numpy
    ! pip install pyqlib
    if "google.colab" in sys.modules:
        # The Google colab environment is a little outdated. We have to downgrade the pyyaml to make it compatible with other packages
        ! pip install pyyaml==5.4.1
    # reload
    site.main()

scripts_dir = Path.cwd().parent.joinpath("scripts")
if not scripts_dir.joinpath("get_data.py").exists():
    # download get_data.py script
    scripts_dir = Path("~/tmp/qlib_code/scripts").expanduser().resolve()
    scripts_dir.mkdir(parents=True, exist_ok=True)
    import requests

    with requests.get("https://raw.githubusercontent.com/microsoft/qlib/main/scripts/get_data.py", timeout=10) as resp:
        with open(scripts_dir.joinpath("get_data.py"), "wb") as fp:
            fp.write(resp.content)

## Step 1 - Setup & data bootstrap

Checks whether the local qlib-format daily bundle exists at `~/.qlib/qlib_data/cn_data`; if missing, downloads it via `scripts/get_data.py`. Then `qlib.init(provider_uri=..., region=REG_CN)` points the data provider at it and applies China-market rules (trade calendar, limit-up/down handling).

If the data is already maintained (e.g. by the daily update automation), the download branch never runs.

In [ ]:
import qlib
import pandas as pd
from qlib.constant import REG_CN
from qlib.utils import exists_qlib_data, init_instance_by_config
from qlib.workflow import R
from qlib.workflow.record_temp import SignalRecord, PortAnaRecord
from qlib.utils import flatten_dict

In [ ]:
# use default data
# NOTE: need to download data from remote: python scripts/get_data.py qlib_data_cn --target_dir ~/.qlib/qlib_data/cn_data
provider_uri = "~/.qlib/qlib_data/cn_data"  # target_dir
if not exists_qlib_data(provider_uri):
    print(f"Qlib data is not found in {provider_uri}")
    sys.path.append(str(scripts_dir))
    from get_data import GetData

    GetData().qlib_data(target_dir=provider_uri, region=REG_CN)
qlib.init(provider_uri=provider_uri, region=REG_CN)

## Step 2 - Pick the universe

`market = "csi300"` = all CSI300 index constituents as the stock pool; `benchmark = "SH000300"` = the CSI300 index itself, used for excess-return comparison in the backtest.

In [ ]:
market = "csi300"
benchmark = "SH000300"

# train model

### Step 3 - Train the model (what the code below does)

The `task` dict has two halves:

- **model**: `LGBModel` (LightGBM regressor) with hand-tuned hyperparameters (`num_leaves=210`, `max_depth=8`, L1/L2 penalties, ...).
- **dataset**: `DatasetH` wrapping an **Alpha158** handler - qlib's built-in pack of ~158 technical factors (K-bar shape, rolling price/volume stats). Window 2008-2020; preprocessing stats (`fit_start_time`/`fit_end_time`) are fit only on 2008-2014 to avoid look-ahead leakage. Split: **train 2008-2014 / valid 2015-2016 / test 2017-2020**.

Everything runs inside `R.start(experiment_name="train_model")` - qlib's MLflow-based experiment recorder: params logged, model fit, trained object saved. `rid` captures the recorder id so this exact model can be reloaded later.

In [ ]:
###################################
# train model
###################################
data_handler_config = {
    "start_time": "2008-01-01",
    "end_time": "2020-08-01",
    "fit_start_time": "2008-01-01",
    "fit_end_time": "2014-12-31",
    "instruments": market,
}

task = {
    "model": {
        "class": "LGBModel",
        "module_path": "qlib.contrib.model.gbdt",
        "kwargs": {
            "loss": "mse",
            "colsample_bytree": 0.8879,
            "learning_rate": 0.0421,
            "subsample": 0.8789,
            "lambda_l1": 205.6999,
            "lambda_l2": 580.9768,
            "max_depth": 8,
            "num_leaves": 210,
            "num_threads": 20,
        },
    },
    "dataset": {
        "class": "DatasetH",
        "module_path": "qlib.data.dataset",
        "kwargs": {
            "handler": {
                "class": "Alpha158",
                "module_path": "qlib.contrib.data.handler",
                "kwargs": data_handler_config,
            },
            "segments": {
                "train": ("2008-01-01", "2014-12-31"),
                "valid": ("2015-01-01", "2016-12-31"),
                "test": ("2017-01-01", "2020-08-01"),
            },
        },
    },
}

# model initialization
model = init_instance_by_config(task["model"])
dataset = init_instance_by_config(task["dataset"])

# start exp to train model
with R.start(experiment_name="train_model"):
    R.log_params(**flatten_dict(task))
    model.fit(dataset)
    R.save_objects(trained_model=model)
    rid = R.get_recorder().id

# prediction, backtest & analysis

### Step 4 - Prediction & backtest (what the code below does)

- **strategy**: `TopkDropoutStrategy` - each day rank all stocks by predicted score, hold the **top 50**, sell the 5 worst-ranked holdings and buy the 5 best-ranked newcomers. Simple, realistic turnover.
- **executor/backtest**: daily simulation, initial capital 100M, with China-market friction: 9.5% price-limit filter, deal at close, 0.05% buy commission, 0.15% sell commission, 5 minimum fee.

Flow: reload trained model by `rid` -> `SignalRecord` saves `pred.pkl` (score for every stock/day in the test window) -> `PortAnaRecord` runs the backtest and stores portfolio metrics.

In [ ]:
###################################
# prediction, backtest & analysis
###################################
port_analysis_config = {
    "executor": {
        "class": "SimulatorExecutor",
        "module_path": "qlib.backtest.executor",
        "kwargs": {
            "time_per_step": "day",
            "generate_portfolio_metrics": True,
        },
    },
    "strategy": {
        "class": "TopkDropoutStrategy",
        "module_path": "qlib.contrib.strategy.signal_strategy",
        "kwargs": {
            "model": model,
            "dataset": dataset,
            "topk": 50,
            "n_drop": 5,
        },
    },
    "backtest": {
        "start_time": "2017-01-01",
        "end_time": "2020-08-01",
        "account": 100000000,
        "benchmark": benchmark,
        "exchange_kwargs": {
            "freq": "day",
            "limit_threshold": 0.095,
            "deal_price": "close",
            "open_cost": 0.0005,
            "close_cost": 0.0015,
            "min_cost": 5,
        },
    },
}

# backtest and analysis
with R.start(experiment_name="backtest_analysis"):
    recorder = R.get_recorder(recorder_id=rid, experiment_name="train_model")
    model = recorder.load_object("trained_model", trusted=True)

    # prediction
    recorder = R.get_recorder()
    ba_rid = recorder.id
    sr = SignalRecord(model, dataset, recorder)
    sr.generate()

    # backtest & analysis
    par = PortAnaRecord(recorder, port_analysis_config, "day")
    par.generate()

# analyze graphs

## Step 5 - Load artifacts & analyze

Four artifacts are pulled from the backtest recorder:

| Artifact | What it is |
|---|---|
| `pred.pkl` | daily predicted score per stock |
| `portfolio_analysis/report_normal_1day.pkl` | daily portfolio returns vs benchmark |
| `portfolio_analysis/positions_normal_1day.pkl` | daily holdings |
| `portfolio_analysis/port_analysis_1day.pkl` | risk/return attribution table |

Charts that follow:

1. **report graph** - cumulative return vs benchmark, drawdown, turnover.
2. **risk analysis** - annualized return, volatility, information ratio, max drawdown, excess-return decomposition.
3. **score IC** - correlation between today's predicted score and next-period actual return (the key signal-quality metric; IC > ~0.05 consistently is decent).
4. **model performance** - grouped returns: do high-ranked stocks actually beat low-ranked ones? Monotonic spread = signal works.

In [ ]:
from qlib.contrib.report import analysis_model, analysis_position
from qlib.data import D

recorder = R.get_recorder(recorder_id=ba_rid, experiment_name="backtest_analysis")
print(recorder)
pred_df = recorder.load_object("pred.pkl")
report_normal_df = recorder.load_object("portfolio_analysis/report_normal_1day.pkl")
positions = recorder.load_object("portfolio_analysis/positions_normal_1day.pkl", trusted=True)
analysis_df = recorder.load_object("portfolio_analysis/port_analysis_1day.pkl")

## analysis position

### report

In [ ]:
analysis_position.report_graph(report_normal_df)

### risk analysis

In [ ]:
analysis_position.risk_analysis_graph(analysis_df, report_normal_df)

## analysis model

In [ ]:
label_df = dataset.prepare("test", col_set="label")
label_df.columns = ["label"]

### score IC

In [ ]:
pred_label = pd.concat([label_df, pred_df], axis=1, sort=True).reindex(label_df.index)
analysis_position.score_ic_graph(pred_label)

### model performance

In [ ]:
analysis_model.model_performance_graph(pred_label)